<a href="https://colab.research.google.com/github/rishiraj333/Democratic_Ranking/blob/main/Project_Topic_Ranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook Summary

### Aim
This notebook aims to aggregate individual student topic rankings into a single, consolidated ranking. It uses a custom `TopicRankAggregator` class that assigns weights to ranks (higher rank = higher weight) and sorts topics based on their total scores. It also handles tie-breaking using lexicographical comparison of rank counts.

### Given Information
*   **`TopicRankAggregator` Class**: A Python class designed to perform the ranking aggregation. It initializes with a mapping of topic IDs to titles and calculates rank weights dynamically based on the total number of topics.
*   **`topic_id_to_title`**: A dictionary containing 10 predefined topics, mapping numerical IDs to their descriptive titles.
*   **`student_rankings`**: A list of 5 individual student rankings. Each ranking is a list of topic IDs, ordered from most preferred to least preferred.

### Usage Guidelines for Reproducibility and Scalability

#### Reproducibility
1.  **Clear Input Documentation**: Ensure that the format and expected content of `topic_id_to_title` and `student_rankings` are well-documented. This notebook already provides clear examples.
2.  **Version Control**: For any changes or extensions, use a version control system (e.g., Git) to track code evolution and ensure consistency.
3.  **Dependency Management**: If external libraries beyond standard Python are used (currently `collections.defaultdict` is standard), list them explicitly (e.g., in a `requirements.txt`).

#### Scalability
1.  **Externalize Inputs**: For a larger number of topics or student rankings, consider loading `topic_id_to_title` and `student_rankings` from external sources like CSV files, JSON files, or databases rather than hardcoding them directly in the notebook.
2.  **Performance Considerations**: The current aggregation logic involves iterating through `num_students * num_topics` and then sorting `num_topics`. For extremely large `num_topics` or `num_students`, this might become a bottleneck. While Python's built-in data structures are optimized, for truly massive datasets, distributed computing frameworks (e.g., Apache Spark) or more specialized data structures might be considered.
3.  **Modularization**: The `TopicRankAggregator` class is already a good step towards modularity. For more complex aggregation logic, further breakdown into smaller, testable functions could be beneficial.

In [ ]:
from collections import defaultdict

class TopicRankAggregator:
    def __init__(self, topic_id_to_title):
        """
        topic_id_to_title: dict
            {topic_id: topic_title}
        """
        self.topic_id_to_title = topic_id_to_title
        self.num_topics = len(topic_id_to_title)

        # Dynamic weight generation (e.g., 10→1)
        self.rank_weights = {
            rank: self.num_topics - rank + 1
            for rank in range(1, self.num_topics + 1)
        }

    def aggregate(self, student_rankings):
        """
        student_rankings: list of lists
            Each inner list contains ALL topic_ids ranked
            from best to worst (length = num_topics).
        """

        total_scores = defaultdict(int)
        rank_counts = defaultdict(lambda: [0]*self.num_topics)

        # --- Validation ---
        for ranking in student_rankings:
            if len(ranking) != self.num_topics:
                raise ValueError("Each student must rank all topics.")
            if len(set(ranking)) != self.num_topics:
                raise ValueError("Duplicate topic detected in ranking.")

        # --- Aggregation ---
        for ranking in student_rankings:
            for rank_position, topic_id in enumerate(ranking, start=1):
                weight = self.rank_weights[rank_position]
                total_scores[topic_id] += weight
                rank_counts[topic_id][rank_position - 1] += 1

        # --- Sorting with lexicographic tie-breaking ---
        sorted_topics = sorted(
            total_scores.keys(),
            key=lambda tid: (
                total_scores[tid],
                *rank_counts[tid]  # unpack rank1 count, rank2 count, ...
            ),
            reverse=True
        )

        # --- Convert IDs to titles ---
        final_ranking = [
            {
                "rank": i + 1,
                "topic_id": tid,
                "title": self.topic_id_to_title[tid],
                "total_score": total_scores[tid]
            }
            for i, tid in enumerate(sorted_topics)
        ]

        return final_ranking

In [ ]:
# Define topics
topic_id_to_title = {
    1: "Action Recognition using Vision Transformers",
    2: "Diabetic Retinopathy Grading",
    3: "Human Faces Generation with Diffusion Model",
    4: "Human Sentiment Analysis",
    5: "Identity-Specific Face Generation with Diffusion Model",
    6: "Melanoma Classification",
    7: "Prompt Tuning Foundation Vision Language Models for Better Downstream Performance",
    8: "Training Deep Learning Models for Industrial Waste Classification",
    9: "Training Vision Transformers for Knee Abnormality Classification",
    10: "Video Anomaly Detection by Weakly Supervised Methods"
}

student_rankings = [
    [7,4,10,5,1,8,9,3,2,6],
    [7,1,10,6,5,8,4,9,2,3],
    [7,10,5,1,4,8,6,9,2,3],
    [7,4,3,1,5,9,8,6,2,10],
    [1,7,5,10,3,9,8,6,2,4]
]

aggregator = TopicRankAggregator(topic_id_to_title)
final_ranking = aggregator.aggregate(student_rankings)

for item in final_ranking:
    print(f"Rank {item['rank']}: {item['title']} (Score: {item['total_score']})")

Rank 1: Prompt Tuning Foundation Vision Language Models for Better Downstream Performance (Score: 49)
Rank 2: Action Recognition using Vision Transformers (Score: 39)
Rank 3: Identity-Specific Face Generation with Diffusion Model (Score: 35)
Rank 4: Video Anomaly Detection by Weakly Supervised Methods (Score: 33)
Rank 5: Human Sentiment Analysis (Score: 29)
Rank 6: Training Deep Learning Models for Industrial Waste Classification (Score: 23)
Rank 7: Training Vision Transformers for Knee Abnormality Classification (Score: 20)
Rank 8: Human Faces Generation with Diffusion Model (Score: 19)
Rank 9: Melanoma Classification (Score: 18)
Rank 10: Diabetic Retinopathy Grading (Score: 10)
